# 🧠 OO-Native Model — Training on Colab

**OO-Native v1** — modèle from scratch propriété du projet OO.

Architecture: OO-SSM (State Space Model custom) + 3 têtes OO (Policy, Pressure, Halt)

**Runtime recommandé: T4 GPU (gratuit)**


In [ ]:
# ── 1. Vérification GPU ──────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Pas de GPU — aller dans Runtime > Changer le type de runtime > T4 GPU')

In [ ]:
# ── 2. Cloner le repo oo-model ───────────────────────────────────────
import os

# Option A: depuis GitHub (si le repo est public)
# !git clone https://github.com/Djiby-diop/oo-model.git
# %cd oo-model

# Option B: uploader le dossier oo-model depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Adapter le chemin selon votre Drive
OO_MODEL_PATH = '/content/drive/MyDrive/oo-model'  # ← modifier si besoin
%cd {OO_MODEL_PATH}
print('CWD:', os.getcwd())

In [ ]:
# ── 3. Installer les dépendances ────────────────────────────────────
!pip install transformers -q
print('✓ Dependencies installed')

In [ ]:
# ── 4. Construire le dataset OO ──────────────────────────────────────
!python scripts/build_dataset.py
import json
with open('data/processed/train.jsonl') as f:
    rows = [json.loads(l) for l in f]
print(f'✓ Dataset: {len(rows)} samples')
print('Exemple:', rows[0])

In [ ]:
# ── 5. Training OO-Native from scratch ──────────────────────────────
# Full training (5000 steps, ~30min sur T4)
!python scripts/train_oo_native.py configs/oo_native_v1.json

In [ ]:
# ── 6. Vérifier le checkpoint ────────────────────────────────────────
import os
ckpt = 'checkpoints/oo-native-v1/oo_native_v1.pt'
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'✓ Checkpoint: {ckpt} ({size_mb:.1f} MB)')
else:
    print('✗ Checkpoint not found')

In [ ]:
# ── 7. Export bare-metal (OONV binary) ──────────────────────────────
!python scripts/export_oo_native.py \
    checkpoints/oo-native-v1/oo_native_v1.pt \
    checkpoints/oo-native-v1/oo_native_v1.bin

# Vérifier la signature magic
with open('checkpoints/oo-native-v1/oo_native_v1.bin', 'rb') as f:
    magic = f.read(4)
    print(f'Magic: {magic} (attendu: b"OONV")')

In [ ]:
# ── 8. Télécharger / sauvegarder sur Drive ───────────────────────────
import shutil

# Sauvegarder sur Google Drive
dst = '/content/drive/MyDrive/oo-model-checkpoints/'
os.makedirs(dst, exist_ok=True)
for f in ['oo_native_v1.pt', 'oo_native_v1.bin', 'tokenizer.json']:
    src = f'checkpoints/oo-native-v1/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✓ Copié: {f} → Drive')

print('\n🎉 Training complete! Modèle OO-Native v1 prêt.')

## Après l'entraînement

1. **Récupérer** `oo_native_v1.pt` + `oo_native_v1.bin` + `tokenizer.json` depuis Drive
2. **Copier** `oo_native_v1.bin` dans `llm-baremetal/models/`
3. **Mettre à jour** `repl.cfg` : `model=oo_native_v1.bin`
4. **Rebuild** UEFI + test QEMU

Pour Kaggle: mêmes étapes, utiliser l'option **Upload Dataset** pour monter le dossier oo-model.